<a href="https://colab.research.google.com/github/dogematos2-sudo/app-tcc-studio/blob/main/tccapp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Instala a biblioteca oficial do Google GenAI
!pip install -q google-genai

# 2. Importa as bibliotecas necessárias
from google import genai
from google.colab import userdata

try:
    # Recupera a chave salva nos Secrets do Colab de forma segura
    api_key = userdata.get('GEMINI_API_KEY')

    # Inicializa o cliente com a chave recuperada
    client = genai.Client(api_key=api_key)

    # Define o tema de arquitetura que você quer pesquisar
    tema_desejado = "Programa de necessidade de IA, legislação e zoneamento. Memorial conceitual e partido."

    print("Iniciando a pesquisa com o Gemini...")

    prompt_completo = f"""
    Atue como um assistente especialista em arquitetura e urbanismo.
    Realize uma pesquisa estruturada e aprofundada sobre o seguinte tema: {tema_desejado}.
    Divida a resposta em tópicos claros: Conceito, Legislação e Zoneamento, Memorial Conceitual e Partido Arquitetônico.
    """

    # Executa a geração de conteúdo usando o modelo recomendado para novos usuários
    response = client.models.generate_content(
        model='gemini-3.8-flash',
        contents=prompt_completo,
    )

    print("\n--- RESULTADO DA PESQUISA ---\n")
    print(response.text)

except Exception as e:
    print("ERRO:", e)
    print("\nPor favor, certifique-se de que cadastrou seu 'GEMINI_API_KEY' no menu lateral de chaves (ícone de chave 🔑) e ativou o acesso deste notebook a ela.")

Iniciando a pesquisa com o Gemini...

--- RESULTADO DA PESQUISA ---

Como especialista em Arquitetura e Urbanismo, preparei uma pesquisa estruturada e aprofundada integrando a temática da Inteligência Artificial (IA) ao desenvolvimento de projetos arquitetônicos e urbanísticos. 

Abaixo, apresento a análise dividida nos quatro tópicos solicitados.

---

### 1. Conceito: O Programa de Necessidades Impulsionado por IA

O **Programa de Necessidades** é a base de qualquer projeto de arquitetura e urbanismo, tradicionalmente formulado por meio de entrevistas, questionários e análise histórica de uso. Com a introdução da Inteligência Artificial, essa etapa evolui de um processo estático para um sistema dinâmico, preditivo e baseado em dados (*data-driven*).

*   **Mineração de Dados e Comportamento:** A IA pode analisar grandes volumes de dados (*Big Data*) sobre como os usuários interagem com espaços semelhantes. Em projetos urbanos, isso inclui padrões de tráfego, densidade populacional e 

In [1]:
from google.colab import drive

drive.mount('/content/drive')
# Salvar arquivos enviados diretamente na pasta do projeto no Drive
# ex: /content/drive/MyDrive/ArchTCC_Projetos/

Mounted at /content/drive


In [3]:
import os

# Caminho da pasta no seu Drive onde os arquivos e imagens ficarão salvos
PASTA_PROJETOS = '/content/drive/MyDrive/ArchTCC_Projetos'

if not os.path.exists(PASTA_PROJETOS):
  os.makedirs(PASTA_PROJETOS)
  print(f'Pasta criada com sucesso em: {PASTA_PROJETOS}')
else:
  print(f'Pasta já existe em: {PASTA_PROJETOS}')

Pasta já existe em: /content/drive/MyDrive/ArchTCC_Projetos


In [4]:
# Exemplo de função para enviar a imagem salva ao Gemini
def analisar_prancha_com_gemini(imagem_path):
  # Carrega a imagem e envia junto ao prompt
  from PIL import Image

  img = Image.open(imagem_path)
  prompt = """
    Atue como um arquiteto especialista. Analise esta prancha de projeto e extraia:
    1. Elementos principais do programa de necessidades visualizado.
    2. Pontos fortes do partido arquitetônico.
    3. Recomendações de melhoria para apresentação técnica.
    """

  response = client.models.generate_content(
      model='gemini-2.5-flash', contents=[img, prompt]
  )
  return response.text

In [5]:
import os
from PIL import Image
import gradio as gr

# Diretórios locais no Hugging Face (ou Colab)
PASTA_EXEMPLOS = "exemplos"
PASTA_UPLOADS = "uploads_usuario"

os.makedirs(PASTA_EXEMPLOS, exist_ok=True)
os.makedirs(PASTA_UPLOADS, exist_ok=True)


def carregar_galeria(pasta):
  """Carrega imagens de uma pasta específica."""
  imagens = []
  if os.path.exists(pasta):
    for arq in sorted(os.listdir(pasta)):
      if arq.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
        caminho = os.path.join(pasta, arq)
        try:
          img = Image.open(caminho)
          imagens.append((img, arq))
        except Exception:
          pass
  return imagens


def processar_upload_usuario(arquivos_novos):
  """Salva as pranchas enviadas pelo usuário e atualiza a galeria dele."""
  if arquivos_novos:
    for file_temp in arquivos_novos:
      nome_arquivo = os.path.basename(file_temp.name)
      caminho_destino = os.path.join(PASTA_UPLOADS, nome_arquivo)
      with open(file_temp.name, "rb") as f_in, open(
          caminho_destino, "wb"
      ) as f_out:
        f_out.write(f_in.read())

  return carregar_galeria(PASTA_UPLOADS)


# Interface Gradio
with gr.Blocks(title="ArchTCC - Estudo e Upload de Pranchas") as app:
  gr.Markdown("""
    # 📐 ArchTCC - Modelos e Upload de Pranchas
    Consulte os modelos de pranchas de exemplo para referência visual e diagramação. Abaixo, envie os seus próprios arquivos para análise.
    """)

  # ABA 1: Pranchas de Exemplo (Modelos)
  with gr.Tab("📌 Pranchas de Exemplo (Modelos)"):
    gr.Markdown(
        "### Referências de Diagramação, Programa de Necessidades e Layout"
    )
    galeria_exemplos = gr.Gallery(
        label="Modelos de Referência", columns=3, height="auto"
    )

  # ABA 2: Upload do Projeto Próprio
  with gr.Tab("📤 Meu Projeto (Upload)"):
    gr.Markdown(
        "### Envie suas pranchas (PNG, JPG) para estruturar seu projeto"
    )
    with gr.Row():
      upload_input = gr.File(
          label="Arraste ou selecione suas pranchas",
          file_count="multiple",
          file_types=["image"],
      )
    btn_upload = gr.Button("Salvar e Visualizar Meu Projeto", variant="primary")

    gr.Markdown("### 🖼️ Galeria do Seu Projeto")
    galeria_usuario = gr.Gallery(
        label="Minhas Pranchas", columns=3, height="auto"
    )

    btn_upload.click(
        fn=processar_upload_usuario,
        inputs=[upload_input],
        outputs=[galeria_usuario],
    )

  # Carrega as galerias na abertura
  app.load(
      fn=lambda: carregar_galeria(PASTA_EXEMPLOS), outputs=[galeria_exemplos]
  )
  app.load(
      fn=lambda: carregar_galeria(PASTA_UPLOADS), outputs=[galeria_usuario]
  )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6bdaaf5259de08fd4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
VITE_GEMINI_API_KEY